# Resources lab

Resources are the ledger for memory slots. They answer: what memory exists, what is free, who is holding it, and what state should the rest of the simulator see?


In [ ]:
from dataclasses import replace

from simyuj.components import PortKind
from simyuj.components.memories import MemoryPositionRecord, MemoryPositionStatus, QuantumMemory
from simyuj.network import Network, Node
from simyuj.network.routing import RoutePlanner
from simyuj.network.topology import NetworkTopology
from simyuj.qstate import SubsystemId
from simyuj.resources import (
    MemoryRef,
    MemorySlotState,
    MemorySlotView,
    Reservation,
    ReservationState,
    ResourceManager,
    memory_refs,
)
from simyuj.resources.route_requirements import (
    requirements_mapping,
    reserve_route_memories,
    route_memory_requirements,
)


## 1. Name memory positions

A `MemoryRef` is the resource-layer address of one memory position.


In [ ]:
alice_refs = memory_refs("alice", "qmem", num_positions=3)

print("Alice memory refs:")
for ref in alice_refs:
    print(" ", ref, "key=", ref.key)


In [ ]:
first_ref = alice_refs[0]
moved_ref = first_ref.with_position(2)

print("Original:", first_ref)
print("Same node/device, different position:", moved_ref)


## 2. Read slot snapshots

A slot view is a snapshot of resource state. It is not the quantum state stored in memory.


In [ ]:
view = MemorySlotView(
    ref=first_ref,
    state=MemorySlotState.FREE,
    ready_at=15,
    expires_at=None,
    metadata=(("link_id", "q_alice_bob"),),
)

print("Slot ref:", view.ref.key)
print("State:", view.state.value)
print("Ready at:", view.ready_at)
print("Metadata:", view.metadata)
print("Available by state alone:", view.is_available)


## 3. Read reservation records

A reservation records ownership intent. State helper methods return new records.


In [ ]:
reservation_record = Reservation(
    reservation_id="reservation:example",
    memory_refs=(first_ref,),
    owner="session-alice-bob",
    link_ids=("q_alice_bob",),
    created_at=10,
    expires_at=40,
    metadata=(("purpose", "demo"),),
)

print("Reservation id:", reservation_record.reservation_id)
print("Owner:", reservation_record.owner)
print("Memory keys:", reservation_record.memory_ref_keys)
print("Links:", reservation_record.link_ids)
print("State:", reservation_record.state.value)
print("Committed copy:", reservation_record.committed().state.value)
print("Original still:", reservation_record.state.value)


## 4. Start a live ledger

Now use the manager. It owns the live ledger.


In [ ]:
manager = ResourceManager()

alice_refs = manager.register_memory(
    "alice",
    "qmem",
    num_positions=3,
    metadata=(("memory_id", "alice.memory"),),
)
relay_refs = manager.register_memory("relay", "qmem", num_positions=2)
bob_refs = manager.register_memory("bob", "qmem", num_positions=2)

print("Registered memory refs:")
for ref in manager.registered_memories():
    print(" ", ref.key)


In [ ]:
def print_slots(title, refs=None):
    print(title)
    selected = manager.registered_memories() if refs is None else refs
    for ref in selected:
        slot = manager.get_slot(ref)
        holder = manager.reservation_for_memory(ref)
        holder_id = None if holder is None else holder.reservation_id
        print(
            f"  {ref.key}: state={slot.state.value}, "
            f"ready_at={slot.ready_at}, holder={holder_id}"
        )


In [ ]:
print_slots("Fresh ledger")
print("Available at tick 0:", [ref.key for ref in manager.available_memories(0)])


## 5. Reserve deterministic slots

Registering is deterministic. Reservations choose the first available refs after sorting.


In [ ]:
reservation = manager.reserve_memories(
    10,
    {"bob": 1, "alice": 2},
    owner="session-1",
    reservation_id="reservation:main",
    expires_at=50,
    metadata=(("route", "alice-bob"),),
)

print("Reservation:", reservation.reservation_id)
print("Selected refs:", reservation.memory_ref_keys)
print("State:", reservation.state.value)
print_slots("After reservation", reservation.memory_refs)


In [ ]:
print("Alice available after reservation:", [ref.key for ref in manager.available_memories(10, "alice")])
print("Bob available after reservation:", [ref.key for ref in manager.available_memories(10, "bob")])


## 6. Commit a reservation

Commit changes reservation state only. It does not mean a qubit has arrived.


In [ ]:
committed = manager.commit_reservation("reservation:main", owner="session-1")

print("Reservation state:", committed.state.value)
print_slots("Slots after commit", committed.memory_refs)


## 7. Mirror physical progress

Marking a slot occupied mirrors physical progress.


In [ ]:
alice_work_slot = committed.memory_refs[0]
manager.mark_occupied(alice_work_slot)

print("Marked occupied:", alice_work_slot.key)
print_slots("After one memory absorbs something", committed.memory_refs)


In [ ]:
released = manager.release_reservation("reservation:main", owner="session-1")

print("Reservation state after release:", released.state.value)
print_slots("After release", released.memory_refs)
print("Held occupied slot now has reservation:", manager.reservation_for_memory(alice_work_slot))


## 8. Free memory deliberately

The occupied slot stayed occupied. Free it only when the physical memory is free again.


In [ ]:
manager.mark_consumed(alice_work_slot)
print_slots("After consuming the occupied slot", (alice_work_slot,))

manager.mark_failed(alice_work_slot)
print_slots("After marking it failed", (alice_work_slot,))

manager.mark_free(alice_work_slot)
print_slots("After repair/reset", (alice_work_slot,))


## 9. Respect recovery time

Ready times matter: a free slot can still be unavailable until hardware recovery time.


In [ ]:
network = Network("resource_scan")
alice = Node("alice")
bob = Node("bob")

alice_memory = QuantumMemory(memory_id="alice.hw", num_positions=2)
alice_memory.positions = (
    replace(alice_memory.positions[0], ready_at=30),
    replace(alice_memory.positions[1], ready_at=5),
)

occupied_position = MemoryPositionRecord(
    position=0,
    status=MemoryPositionStatus.OCCUPIED,
    memory_subsystem=SubsystemId("memory:bob.hw:position:0"),
    stored_time=4,
    last_noise_update_time=4,
    expires_at=90,
)
bob_memory = QuantumMemory(memory_id="bob.hw", num_positions=1)
bob_memory.positions = (occupied_position,)

alice.add_device("qmem", alice_memory)
bob.add_device("qmem", bob_memory)
alice.add_device("not_memory", object())

network.add_node(alice)
network.add_node(bob)


In [ ]:
scanned = ResourceManager.from_network(network)

print("Scanned refs:")
for ref in scanned.registered_memories():
    slot = scanned.get_slot(ref)
    print(
        f"  {ref.key}: state={slot.state.value}, "
        f"ready_at={slot.ready_at}, expires_at={slot.expires_at}, metadata={slot.metadata}"
    )


In [ ]:
print("Alice available at tick 10:", [ref.key for ref in scanned.available_memories(10, "alice")])
print("Alice available at tick 30:", [ref.key for ref in scanned.available_memories(30, "alice")])
print("Bob available at tick 30:", [ref.key for ref in scanned.available_memories(30, "bob")])


## 10. Ask for a device pool

Device-targeted requirements let a caller ask for specific memory pools.


In [ ]:
targeted = ResourceManager()
targeted.register_memory("relay", "near_fiber", num_positions=2, metadata=(("link_id", "q_alice_relay"),))
targeted.register_memory("relay", "far_fiber", num_positions=2, metadata=(("link_id", "q_relay_bob"),))

print("All relay refs:", [ref.key for ref in targeted.available_memories(0, "relay")])
print("Near-fiber refs:", [ref.key for ref in targeted.available_memories(0, "relay", link_id="q_alice_relay")])


In [ ]:
targeted_reservation = targeted.reserve_memories(
    0,
    {"relay": {"near_fiber": 1, "far_fiber": 2}},
    owner="session-2",
)

print("Targeted reservation:", targeted_reservation.reservation_id)
print("Selected refs:", targeted_reservation.memory_ref_keys)


## 11. Derive route requirements

Route helpers keep policy outside the resource layer. You decide how many slots each node needs.


In [ ]:
route_network = Network("route_resources")
for node_id in ("alice", "relay", "bob"):
    route_network.add_node(Node(node_id))

route_network.add_quantum_link("q_alice_relay", "alice", "relay")
route_network.add_quantum_link("q_relay_bob", "relay", "bob")

route = route_network.fewest_hops_path("alice", "bob", port_kind=PortKind.QUANTUM)

print("Route nodes:", route.node_ids)
print("Route links:", route.link_ids)


In [ ]:
def one_at_endpoints_two_at_relay(node_id, index, route_length):
    if index == 0 or index == route_length - 1:
        return 1
    return 2


requirements = route_memory_requirements(
    route,
    node_requirements=one_at_endpoints_two_at_relay,
)

print("Route requirements:")
for requirement in requirements:
    print(" ", requirement.node_id, "needs", requirement.requirement)

print("As manager mapping:", requirements_mapping(requirements))


In [ ]:
route_manager = ResourceManager()
route_manager.register_memory("alice", "qmem", num_positions=1)
route_manager.register_memory("relay", "qmem", num_positions=2)
route_manager.register_memory("bob", "qmem", num_positions=1)

route_reservation = reserve_route_memories(
    100,
    route_manager,
    route,
    node_requirements=one_at_endpoints_two_at_relay,
    owner="session-route",
    reservation_id="reservation:route",
    metadata=(("selected_route", route.link_ids),),
)

print("Reserved for route:", route_reservation.reservation_id)
print("Refs:", route_reservation.memory_ref_keys)
print("Metadata:", route_reservation.metadata)


In [ ]:
print("Route manager ledger:")
for ref in route_manager.registered_memories():
    slot = route_manager.get_slot(ref)
    print(f"  {ref.key}: {slot.state.value}")


## 12. Expire stale holds

Reservations can expire. Slots still reserved become free; occupied slots stay occupied.


In [ ]:
expiry_manager = ResourceManager()
expiry_refs = expiry_manager.register_memory("alice", "qmem", num_positions=3)

early = expiry_manager.reserve_memory_refs(0, (expiry_refs[0],), owner="session", expires_at=10)
busy = expiry_manager.reserve_memory_refs(0, (expiry_refs[1],), owner="session", expires_at=10)
later = expiry_manager.reserve_memory_refs(0, (expiry_refs[2],), owner="session", expires_at=20)
expiry_manager.mark_occupied(busy.memory_refs[0])

expired = expiry_manager.expire_before(10)

print("Expired reservations:", [(res.reservation_id, res.state.value) for res in expired])
for ref in expiry_refs:
    print(ref.key, expiry_manager.get_slot(ref).state.value)


In [ ]:
print("Later reservation state:", expiry_manager.get_reservation(later.reservation_id).state.value)
print("Busy reservation holder removed:", expiry_manager.reservation_for_memory(busy.memory_refs[0]))


## Keep this model in your head

Keep the boundary in mind: resources decide what can be held. Components and control decide what actually happens to the memory.
